### Ingest Fact Data into Bronze Layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
order_items_schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
])

Incremental load with Auto Loader: each run ingests only the files in `landing/` that were not ingested by a previous run and appends them to the bronze table.
Ingested files are tracked in the checkpoint, so new daily files must be uploaded with a new file name (e.g. `order_items_2025-11-01.csv`).

In [0]:
landing_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/"
checkpoint_path = "/Volumes/ecommerce/source_data/checkpoints/brz_order_items"
bronze_table = f"{catalog_name}.bronze.brz_order_items"

In [0]:
# No checkpoint means bronze was never loaded incrementally (old full-load table) or the checkpoint was reset.
# Drop the table so Auto Loader rebuilds it from every file in landing/ instead of appending duplicates.
try:
    dbutils.fs.ls(checkpoint_path)
    checkpoint_exists = True
except Exception as e:
    if "FileNotFound" not in str(e) and "NOT_FOUND" not in str(e):
        raise
    checkpoint_exists = False

if not checkpoint_exists:
    print(f"No checkpoint found, rebuilding {bronze_table} from all files in {landing_path}")
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")

In [0]:
# Load only new files using the schema defined
df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("pathGlobFilter", "*.csv") \
    .schema(order_items_schema) \
    .load(landing_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [0]:
# Append new files to the bronze layer and stop once all available files are processed
query = df.writeStream \
    .option("checkpointLocation", checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(bronze_table)

query.awaitTermination()

In [0]:
# Sanity check: rows per ingested file (latest first)
display(spark.table(bronze_table).groupBy("file_name").count().orderBy(F.col("file_name").desc()).limit(10))